In [70]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient , command , Input , Output
from azure.ai.ml.entities import Data , Environment , Model , ComputeInstance , AmlCompute ,  BatchDeployment , ModelBatchDeployment
from azure.ai.ml.constants import AssetTypes

from datetime import datetime
from zoneinfo import ZoneInfo


from azure.ai.ml.dsl import pipeline
import pandas as pd 
import numpy as np


In [2]:
from azure.ai.ml import MLClient
from azure.ai.ml.dsl import pipeline
from azure.ai.ml.entities import (
    BatchEndpoint,
    BatchDeployment,
    BatchRetrySettings
)
from azure.identity import DefaultAzureCredential , AzureCliCredential


In [3]:

# Connect mlClient
credential = DefaultAzureCredential()

# ml_client = MLClient.from_config(
#     credential=DefaultAzureCredential()
# )

ml_client = MLClient.from_config(
    credential=AzureCliCredential()
)



Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [4]:

# ml_client = MLClient(
#     credential=credential,
#     subscription_id = ml_client.subscription_id,
#     resource_group_name = ml_client.resource_group_name,
#     workspace_name = ml_client.workspace_name
# )

ml_client = MLClient(
    credential=credential,
    subscription_id = '49297e8f-2e51-410c-aee9-69db39628913',
    resource_group_name = 'dp-100_trail',
    workspace_name = 'dp-100'
)



Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [5]:
env_name = "Pipeline_Flight-env"
exp_name = 'score_and_create_endpoint'
conda_file = "Environment.yml"
image_details ="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"
DataStore_name = 'flightsdatastrore'
Blob_input = 'flightdelaydata.csv'

compute_instance_name = 'CPU432'

Prepare_data = "Prepare_date.py"
Train_data = 'Train_data.py'

In [6]:
print("Subscription ID:", ml_client.subscription_id)
print("Resource Group:", ml_client.resource_group_name)
print("Workspace Name:", ml_client.workspace_name)

Subscription ID: 49297e8f-2e51-410c-aee9-69db39628913
Resource Group: dp-100_trail
Workspace Name: dp-100


In [7]:

# get data set
datastore = ml_client.datastores.get(DataStore_name)
print(datastore)
path = f"azureml://subscriptions/{ml_client.subscription_id}/resourcegroups/{ml_client.resource_group_name}/workspaces/{ml_client.workspace_name}/datastores/{datastore.name}/paths/{Blob_input}"


account_name: dp100n
container_name: flightcontainer
credentials: {}
endpoint: core.windows.net
id: /subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/datastores/flightsdatastrore
name: flightsdatastrore
protocol: https
tags: {}
type: azure_blob



In [8]:

# Create environment
env = Environment(
    name = f"{env_name}",
    description="Custom AML environment",
    conda_file=f"{conda_file}",
    image = f"{image_details}" 
    # version = '1'
)

In [9]:


# Register environment
ml_client.environments.create_or_update(env)

# print("Environment created successfully")

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'Pipeline_Flight-env', 'description': 'Custom AML environment', 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/environments/Pipeline_Flight-env/versions/2', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpu432/code/Users/brenjoelsikha/Pipeline_Project_with_batch_deployment', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x759ef0eccaf0>, 'serialize': <msrest.serialization.Serializer object at 0x759ef0eceb90>, 'version': '2', 'conda_file': {'channels': ['conda-forge'], 'depe

In [10]:




# # Register the original data set  URI FILE
# raw_data = Data(
#     path=path,
#     type=AssetTypes.URI_FILE,
#     name=DataAsset_Name,
#     # version="Main",
#     description="Raw flight delay dataset"
# )

# ml_client.data.create_or_update(raw_data)

In [11]:
# # Create a compute instance

# compute_instance_name = "test-my-compute-instance"
# compute_instance = ComputeInstance(
#     name=compute_instance_name ,
#     size="STANDARD_DS3_V2",
# )

# ml_client.begin_create_or_update(compute_instance).result()

# print("Compute instance created")

In [12]:

# # Create compute cluster 
# compute_cluster_name = 'testCluster'
# cpu_cluster = AmlCompute(
#     name=compute_cluster_name,
#     type="amlcompute",
#     size="STANDARD_DS2_V2",
#     min_instances=0,
#     max_instances=1,
#     idle_time_before_scale_down=120,
# )

# ml_client.begin_create_or_update(cpu_cluster).result()

# print("Compute cluster created")

In [13]:


# Step 1 - Prepare Data
prepare_data_step = command(
    name= 'prepare_data',
    display_name="Prepare Data",

    code="./src",

    command=f"""
    python {Prepare_data} \
    --input_data ${{{{inputs.input_data}}}} \
    --prepped_data ${{{{outputs.prepped_data}}}}
    """,

    inputs={
        "input_data": Input(
            type=AssetTypes.URI_FILE,
            path=path
        )
    },

    outputs={
        "prepped_data": Output(
            type=AssetTypes.URI_FOLDER
        )
    },

    environment=f'{env_name}@latest',
    compute=compute_instance_name
)

# Step 2 - Train Model
train_model_step = command(
    name='train_data',
    display_name="Train Model",

    code="./src",

    command=f"""
    python {Train_data} \
    --prepped_data ${{{{inputs.prepped_data}}}} \
    --model_output ${{{{outputs.model_output}}}}
    """,

    inputs={
        "prepped_data": Input(
            type=AssetTypes.URI_FOLDER
        ),
    },
    outputs = { 
        'model_output' : Output(
            type=AssetTypes.URI_FOLDER
            ),
        },
    environment=f'{env_name}@latest',  # "pipeline_env@latest",
    compute=compute_instance_name
)


In [14]:

# Build Pipeline
@pipeline()
def training_pipeline():

    # Step 1 execution
    prep_step = prepare_data_step()
    prep_step.compute=compute_instance_name

    # Step 2 execution with output from step 1
    train_step = train_model_step(
        prepped_data=prep_step.outputs.prepped_data
    )
    train_step.compute = compute_instance_name

    #step 3 execution


    return {
        "prepared_data": prep_step.outputs.prepped_data,
        "model_output": train_step.outputs.model_output
    }


In [15]:

# Create pipeline job
pipeline_job = training_pipeline()


# Experiment name
pipeline_job.experiment_name = exp_name

# Equivalent of regenerate_outputs=True
pipeline_job.settings.force_rerun = True



In [16]:
# Submit pipeline
returned_job = ml_client.jobs.create_or_update(pipeline_job)

ml_client.jobs.stream(returned_job.name)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
pathOnCompute is not a known attribute

RunId: amusing_owl_t5flw2d3pw
Web View: https://ml.azure.com/runs/amusing_owl_t5flw2d3pw?wsid=/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourcegroups/dp-100_trail/workspaces/dp-100

Streaming logs/azureml/executionlogs.txt

[2026-06-01 07:59:19Z] Submitting 1 runs, first five are: 123abd23:a7d71da1-8ab6-4aab-9405-3b20005a1920
[2026-06-01 08:00:06Z] Completing processing run id a7d71da1-8ab6-4aab-9405-3b20005a1920.
[2026-06-01 08:00:06Z] Submitting 1 runs, first five are: 73989112:11e6330e-dd8e-4058-9a25-5c5125da814b
[2026-06-01 08:00:39Z] Completing processing run id 11e6330e-dd8e-4058-9a25-5c5125da814b.

Execution Summary
RunId: amusing_owl_t5flw2d3pw
Web View: https://ml.azure.com/runs/amusing_owl_t5flw2d3pw?wsid=/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourcegroups/dp-100_trail/workspaces/dp-100



In [17]:
# # # Stop compute instance
# ml_client.compute.begin_stop(compute_instance_name).wait()

# print("Stopped")

In [18]:
# # # Delete compute instance
# ml_client.compute.begin_delete(compute_instance_name).wait()

# print("Compute instance deleted")

In [19]:
# Delete Compute Cluster 
# ml_client.compute.begin_delete(compute_cluster_name).wait()

In [20]:
# from azure.identity import InteractiveBrowserCredential

# auth_credential = InteractiveBrowserCredential()

# token = credential.get_token(
#     "https://ml.azure.com/.default"
# )

# access_token = token.token

In [21]:
# import requests
# import json

# exp_name = "pipeline_endpoint_exp"
# endpoint = "https://eastus2.api.azureml.ms/pipelines/v1.0/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/PipelineRuns/PipelineEndpointSubmit/Id/6bb32fbc-2a8f-4c1e-8464-24bc50f00171"

# headers = {
#     "Authorization": f"Bearer {access_token}",
#     "Content-Type": "application/json"
# }

# payload = {
#     "ExperimentName": "PipelineTrigger"
# }

# response = requests.post(
#     endpoint,
#     headers=headers,
#     json=payload
# )

# print(response.status_code)
# print(response.text)

## 2


## 3

In [22]:
# # ---------------------------------------------------
# # CREATE PIPELINE JOB
# # ---------------------------------------------------

# pipeline_job = training_pipeline()

# pipeline_job.settings.force_rerun = True


# # ---------------------------------------------------
# # CREATE BATCH ENDPOINT
# # ---------------------------------------------------

# from azure.ai.ml.entities import (
#     BatchEndpoint,
#     PipelineComponentBatchDeployment
# )

# endpoint_name = "training-pipeline-endpoint"

# endpoint = BatchEndpoint(
#     name=endpoint_name,
#     description="Training pipeline endpoint"
# )

# ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

# print("Endpoint created")


# # ---------------------------------------------------
# # CREATE DEPLOYMENT
# # ---------------------------------------------------

# deployment = PipelineComponentBatchDeployment(
#     name="training-deployment",
#     endpoint_name=endpoint_name,

#     component=pipeline_job.component
# )

# ml_client.batch_deployments.begin_create_or_update(deployment).result()

# print("Deployment created")


# # ---------------------------------------------------
# # SET DEFAULT DEPLOYMENT
# # ---------------------------------------------------

# endpoint = ml_client.batch_endpoints.get(endpoint_name)

# endpoint.defaults.deployment_name = "training-deployment"

# ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

# print("Default deployment set")


# # ---------------------------------------------------
# # INVOKE ENDPOINT
# # ---------------------------------------------------

# job = ml_client.batch_endpoints.invoke(
#     endpoint_name=endpoint_name
# )

# print(job)
# ml_client.jobs.stream(
#     "pipelinejob-e71fd4a1-7ee1-4053-8afd-8a29f3a34f56"
# )

In [23]:
print(returned_job.name)

amusing_owl_t5flw2d3pw


In [24]:

child_jobs = ml_client.jobs.list(parent_job_name=returned_job.name)

train_job_id = None

for job in child_jobs:
    if job.display_name == "train_step":
        train_job_id = job.name
        break

print("Train Job ID:", train_job_id)

model = Model(
    path=f"azureml://jobs/{train_job_id}/outputs/model_output/paths/model.pkl", #azureml/1276ed32-4680-4539-a2af-9a31aad4a2c1/model_output/
    # path=returned_job.outputs.model_output.path,
    name="flight-delay-model",
    type="custom_model"
)

registered_model = ml_client.models.create_or_update(model)




Train Job ID: 11e6330e-dd8e-4058-9a25-5c5125da814b


In [25]:
print(returned_job.outputs)

{'prepared_data': <azure.ai.ml.entities._job.pipeline._io.base.PipelineOutput object at 0x759ef0ef1450>, 'model_output': <azure.ai.ml.entities._job.pipeline._io.base.PipelineOutput object at 0x759ef0ef2ad0>}


In [26]:
completed_job = ml_client.jobs.get(returned_job.name)

print(completed_job.outputs)
print(completed_job.outputs.model_output.path)

{'prepared_data': <azure.ai.ml.entities._job.pipeline._io.base.PipelineOutput object at 0x759e9d5ed300>, 'model_output': <azure.ai.ml.entities._job.pipeline._io.base.PipelineOutput object at 0x759e9d5efd90>}
None


In [27]:
child_jobs = ml_client.jobs.list(parent_job_name=returned_job.name)

for job in child_jobs:
    print(job.name, job.display_name)

a7d71da1-8ab6-4aab-9405-3b20005a1920 prep_step
11e6330e-dd8e-4058-9a25-5c5125da814b train_step


In [28]:
print(registered_model.name)

flight-delay-model


In [29]:
len("FD-batch-endpoint01-06-2026-1208")

32

## Create online endpoint


In [54]:

suffix = "-01-06-2026-1340"
endpoint_name= "FD-endpoint" + suffix
scoreing_script= "score.py"
# input_data = "./data/1.csv"
input_data = "./data/1.csv"
deployment_name="FD-deployment" + suffix


In [55]:
endpoint_name

'FD-endpoint-01-06-2026-1340'

In [56]:

endpoint = BatchEndpoint(
    name=endpoint_name,
    description="Flight delay prediction batch endpoint"
)

ml_client.batch_endpoints.begin_create_or_update(endpoint).result()

BatchEndpoint({'scoring_uri': 'https://fd-endpoint-01-06-2026-1340.eastus2.inference.ml.azure.com/jobs', 'openapi_uri': None, 'provisioning_state': 'Succeeded', 'name': 'fd-endpoint-01-06-2026-1340', 'description': 'Flight delay prediction batch endpoint', 'tags': {}, 'properties': {'BatchEndpointCreationApiVersion': '2023-10-01', 'azureml.onlineendpointid': '/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/batchEndpoints/fd-endpoint-01-06-2026-1340'}, 'print_as_yaml': False, 'id': '/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/batchEndpoints/fd-endpoint-01-06-2026-1340', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpu432/code/Users/brenjoelsikha/Pipeline_Project_with_batch_deployment', 'creation_context': None, 'serialize': <msrest.serialization.Seriali

In [57]:
from azure.ai.ml.entities import BatchDeployment , ModelBatchDeployment

deployment = ModelBatchDeployment(
    name=deployment_name,
    endpoint_name=endpoint_name,
    model=registered_model,
    compute=compute_instance_name,
    environment=env,
    code_path="./src",
    scoring_script=scoreing_script
)

ml_client.batch_deployments.begin_create_or_update(deployment).result()

BatchDeployment({'provisioning_state': 'Succeeded', 'endpoint_name': 'fd-endpoint-01-06-2026-1340', 'type': None, 'name': 'fd-deployment-01-06-2026-1340', 'description': None, 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/batchEndpoints/fd-endpoint-01-06-2026-1340/deployments/fd-deployment-01-06-2026-1340', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpu432/code/Users/brenjoelsikha/Pipeline_Project_with_batch_deployment', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x759e96d1d630>, 'serialize': <msrest.serialization.Serializer object at 0x759e96d1f010>, 'model': '/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourceGroups/dp-100_trail/providers/Microsoft.MachineLearningServices/workspaces/dp-100/models/flight-delay-model/versions/11', 'code_config

In [58]:
endpoint = ml_client.batch_endpoints.get(
    endpoint_name
)

print(endpoint.name)

fd-endpoint-01-06-2026-1340


In [59]:
for deployment in ml_client.batch_deployments.list(endpoint_name):
    print(deployment.name)

fd-deployment-01-06-2026-1340


In [60]:
job = ml_client.batch_endpoints.invoke(
    endpoint_name=endpoint_name,
    deployment_name=deployment.name,
    inputs={
        "input_data": Input(path=input_data)
    }
)



In [61]:
print(job.name)

batchjob-d9bacb18-4e8f-4a74-8da8-02ee5b4e127d


In [62]:
import os
print(os.getcwd())

/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpu432/code/Users/brenjoelsikha/Pipeline_Project_with_batch_deployment


In [63]:
input_data = os.path.join(os.getcwd(), "data", "1.csv")

print(input_data)
print(os.path.exists(input_data))



/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpu432/code/Users/brenjoelsikha/Pipeline_Project_with_batch_deployment/data/1.csv
True


In [64]:
job = ml_client.batch_endpoints.invoke(
    endpoint_name=endpoint_name,
    deployment_name=deployment.name,
    input=Input(
        type="uri_file",
        path=input_data
    )
)

In [65]:
ml_client.jobs.stream(job.name)

RunId: batchjob-51590458-35f3-4ef8-a3f0-033cf6689451
Web View: https://ml.azure.com/runs/batchjob-51590458-35f3-4ef8-a3f0-033cf6689451?wsid=/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourcegroups/dp-100_trail/workspaces/dp-100

Streaming logs/azureml/executionlogs.txt

[2026-06-01 08:13:21Z] Submitting 1 runs, first five are: 76721330:adcd8006-5ed3-4839-a3e7-1b30cbfb99d4
[2026-06-01 08:14:44Z] Completing processing run id adcd8006-5ed3-4839-a3e7-1b30cbfb99d4.

Execution Summary
RunId: batchjob-51590458-35f3-4ef8-a3f0-033cf6689451
Web View: https://ml.azure.com/runs/batchjob-51590458-35f3-4ef8-a3f0-033cf6689451?wsid=/subscriptions/49297e8f-2e51-410c-aee9-69db39628913/resourcegroups/dp-100_trail/workspaces/dp-100



In [66]:
ml_client.jobs.download(
    name=job.name,
    download_path="./outputs"
)

In [67]:
import os

print(os.listdir("outputs"))

['predictions.csv', 'predictions_b2c73b7a549c4dd9841f7205e1c68064.csv']


In [71]:
df1 = pd.read_csv("outputs/predictions.csv")
df2 = pd.read_csv("outputs/predictions_b2c73b7a549c4dd9841f7205e1c68064.csv")

print(df1.shape)
print(df2.shape)

(0, 1)
(100, 15)


In [73]:
df2.head()

,Year,Month,DayofMonth,DayOfWeek,Carrier,OriginAirportID,DestAirportID,CRSDepTime,DepDelay,DepDel15,CRSArrTime,ArrDelay,ArrDel15,Cancelled,prediction
0,2013,4,19,5,DL,11433,13303,837,-3.0,0.0,1138,1.0,0,0,-6.992842
1,2013,4,19,5,DL,14869,12478,1705,0.0,0.0,2336,-8.0,0,0,-3.129275
2,2013,4,19,5,DL,14057,14869,600,-4.0,0.0,851,-15.0,0,0,-7.764663
3,2013,4,19,5,DL,15016,11433,1630,28.0,1.0,1903,24.0,1,0,25.816608
4,2013,4,19,5,DL,11193,12892,1615,-6.0,0.0,1805,-11.0,0,0,-9.880161


In [74]:
df = df2.copy()

In [76]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_true = df["ArrDelay"]
y_pred = df["prediction"]


In [78]:

# print("Accuracy:", accuracy_score(y_true, y_pred))
# print("Precision:", precision_score(y_true, y_pred, average='weighted'))
# print("Recall:", recall_score(y_true, y_pred, average='weighted'))
# print("F1 Score:", f1_score(y_true, y_pred, average='weighted'))

# print("\nClassification Report:\n")
# print(classification_report(y_true, y_pred))

# print("\nConfusion Matrix:\n")
# print(confusion_matrix(y_true, y_pred))

In [80]:
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np
# Metrics
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE : 9.547274490357793
MSE : 151.621485127035
RMSE: 12.313467632110582
R2 Score: 0.89375475434266
